In [ ]:
from warnings import filterwarnings

import pandas as pd
import plotly.express as px
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    PolynomialFeatures,
    StandardScaler,
)

filterwarnings("ignore")

In [ ]:
df = pd.read_csv("data/data.csv", index_col=0)

In [ ]:
df.sample(5)

In [ ]:
df.isna().mean().round(2)
# No missing data

In [ ]:
fig = px.histogram(
    df,
    x="Absenteeism time in hours",
    title="Majority of workers are absent 5 hours or less",
)
fig.write_image("images/hours_missing.png")

In [ ]:
# create a field for predicting employees that miss more than 5 hours
df["miss_more_than_5_hours"] = df["Absenteeism time in hours"].map(
    lambda hr: 1 if hr > 5 else 0
)
df = df.reset_index()
df.drop(["Absenteeism time in hours", "ID"], inplace=True, axis=1)

In [ ]:
fig = px.pie(
    df, "miss_more_than_5_hours", title="37% of workers miss more than 5 hours"
)
fig.write_image("images/worker_hours.png")

In [ ]:
X = df.drop("miss_more_than_5_hours", axis=1)
y = df["miss_more_than_5_hours"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, test_size=0.2
)

In [ ]:
pipe = Pipeline(
    [
        ("std", StandardScaler()),
        ("poly", PolynomialFeatures()),
        ("log", LogisticRegression()),
    ]
)

grid = GridSearchCV(pipe, param_grid={"poly__degree": [2, 3, 4]}, verbose=3)

grid.fit(X_train, y_train)

In [ ]:
grid.best_params_

In [ ]:
f1 = f1_score(y_test, grid.best_estimator_.predict(X_test))
recall = recall_score(y_test, grid.best_estimator_.predict(X_test))
precision = precision_score(y_test, grid.best_estimator_.predict(X_test))
accuracy = accuracy_score(y_test, grid.best_estimator_.predict(X_test))
print(f"F1 Score: {f1:.3f}")
print(f"Recall: {recall:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Accuracy: {accuracy:.3f}")

In [ ]:
confusion_matrix(y_test, grid.best_estimator_.predict(X_test))